# Hotel Booking Data Analysis with Pandas

Exploratory data analysis on a real hotel booking dataset — 119,390 bookings across two hotel types.

**Covers:**
- Data loading, shape inspection, null detection
- Targeted column removal and cleaning
- Filtering with multiple boolean conditions
- Aggregation: means, value counts, grouped statistics
- Derived columns (total nights, total cost)
- String extraction (last name parsing)
- Date range filtering with `query()`

**Dataset:** Hotel booking demand dataset (Kaggle) — city hotel + resort hotel bookings, 2015–2017.

In [ ]:
import pandas as pd

---
## 1. Load and Inspect

In [ ]:
df = pd.read_csv('data/hotel_booking_data.csv')
df.head(4)

In [ ]:
print(f"Shape: {df.shape}  →  {df.shape[0]:,} bookings × {df.shape[1]} features")
print(f"\nFeatures ({df.shape[1]}):")
print(df.columns.tolist())

---
## 2. Missing Data

Identifying which columns have nulls and sorting by severity.

In [ ]:
null_counts = df.isnull().sum().sort_values(ascending=False)
null_pcts   = (null_counts / len(df) * 100).round(2)

missing_report = pd.DataFrame({'null_count': null_counts, 'pct_missing': null_pcts})
missing_report[missing_report.null_count > 0]

---
## 3. Column Types and Value Distributions

In [ ]:
df.info()

In [ ]:
print("Hotel types:")
print(df['hotel'].value_counts())
print()
print("Cancellation rate:")
print(df['is_canceled'].value_counts(normalize=True).round(3))

---
## 4. Drop High-Null Columns

`company` (94% null) and `agent` (13% null) are removed — both are identifiers that add noise rather than signal for analysis.

In [ ]:
df_clean = df.drop(['company', 'agent'], axis=1)

print(f"Columns before: {df.shape[1]}  →  after: {df_clean.shape[1]}")
print(f"Remaining nulls: {df_clean.isnull().sum().sum()}")

---
## 5. Average Daily Rate — Resort Hotel, Non-Cancelled, No Children/Babies

Three simultaneous filter conditions using bitwise `&`. Note: parentheses around each condition are required in pandas boolean indexing.

In [ ]:
filtered = df_clean[
    (df_clean['children']    == 0) &
    (df_clean['is_canceled'] == 0) &
    (df_clean['hotel']       == 'Resort Hotel')
]

avg_adr = filtered['adr'].mean()
print(f"Bookings matching filter : {len(filtered):,}")
print(f"Average daily rate (ADR) : ${avg_adr:.2f}")

---
## 6. Top 5 Most Common Country Codes

In [ ]:
df['country'].value_counts().head(5)

---
## 7. Overall Mean ADR

In [ ]:
mean_adr = round(df['adr'].mean(), 2)
print(f"Mean ADR across all bookings: ${mean_adr}")

---
## 8. Average Total Nights per Stay

Creating a derived column — total nights = weekend nights + weekday nights.

In [ ]:
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']

avg_nights = round(df['total_nights'].mean(), 2)
print(f"Average total nights per stay: {avg_nights}")

---
## 9. Average Total Cost per Stay

Total cost = ADR × total nights. This is a per-row calculation — NumPy broadcasting handles the element-wise multiply across 119K rows instantly.

In [ ]:
total_cost_mean = round(
    (df['adr'] * df['total_nights']).mean(), 2
)
print(f"Average total cost per stay: ${total_cost_mean}")

---
## 10. Guests with Exactly 5 Special Requests

In [ ]:
df.loc[
    df['total_of_special_requests'] == 5,
    ['name', 'email']
].drop_duplicates()

---
## 11. Top 5 Most Common Last Names

Chaining string methods: `strip()` removes whitespace, `split()` tokenises on spaces, `str[-1]` extracts the last token (last name), then `value_counts()` counts frequency.

In [ ]:
top_lastnames = (
    df['name']
    .str.strip()
    .str.split()
    .str[-1]
    .value_counts()
    .head(5)
)
print("Top 5 last names:")
print(top_lastnames)

---
## 12. Arrivals on Days 1–15 of the Month

Two equivalent approaches shown: `query()` (SQL-style string syntax) and boolean mask. `query()` is more readable for simple numeric comparisons.

In [ ]:
# Approach 1: query() — clean and readable
count_query = df.query('1 <= arrival_date_day_of_month <= 15').shape[0]

# Approach 2: boolean mask — more explicit
count_mask = (
    (df['arrival_date_day_of_month'] >= 1) &
    (df['arrival_date_day_of_month'] <= 15)
).sum()

print(f"Arrivals on days 1–15: {count_query:,}")
assert count_query == count_mask, "Methods disagree!"
print(f"Fraction of all bookings: {count_query / len(df):.1%}")